# Fair module test on the FULL data (6 GPU-hours budget, GPU T4 x2)

Earlier pilots used one-third of the data for 30 epochs (~9,000 steps). Here both models train on **all 15,000
training images** for 16 epochs (~15,000 steps), which gives a module a fairer chance to switch on.

* **GPU 0 – `yolo26s-p2` + "hidden people count extra"** (reference)
* **GPU 1 – `yolo26s-p2` + Visible-Body Attention (neck) + "hidden people count extra"** (candidate)

Expected time: ~30 min data preparation + ~3.7 h training + ~15 min scoring ≈ **4.5 h**. An emergency brake stops
training after 4.3 h if it runs slow, so the run cannot exceed the 6-hour quota.

**Setup**: GPU T4 x2 · Internet on · Add Input: **CrowdHuman** only. Then *Save Version → Save & Run All*.
The results are written to `results/summary.json`.

In [ ]:
import glob, os, json
_f = glob.glob("/kaggle/input/**/annotation_train.odgt", recursive=True)
assert _f, "CrowdHuman not found: add it with 'Add Input'"
CH_ROOT = os.path.dirname(_f[0])
print("CH_ROOT =", CH_ROOT)
EPOCHS, FRACTION, IMGSZ, BATCH = 16, 1.0, 640, 16     # full data
MAX_HOURS, CLOSE_MOSAIC = 4.3, 5                       # emergency brake; last 5 epochs without mosaic
WORK = "/kaggle/working"
DATA = "/tmp/ch_yolo_heads"

In [ ]:
!pip -q install ultralytics==8.4.163
!git clone -q --depth 1 https://github.com/Purkialo/CrowdDet.git /kaggle/working/CrowdDet || true
import torch, ultralytics
print(ultralytics.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py — convert CrowdHuman and CityPersons to YOLO format for VG-YOLO.

What it does
  * writes one YOLO *pose-format* label file per image:
        <vis_bin> cx cy w h  vx1 vy1 2  vx2 vy2 2  hx1 hy1 f  hx2 hy2 f   (head box; f = 2 if labelled, else 0)
      - cx cy w h      : full-body box, clipped to the image, normalised
      - vx1 vy1, vx2 vy2: top-left and bottom-right corners of the visible-body box, stored as two
                          "keypoints" so that mosaic / flip / scale move them together with the box
      - vis_bin        : the person's occlusion visibility v quantised into NBINS bins.
        The custom trainer maps it back to (class 0, v) after augmentation.
  * optionally paints "ignore" regions (crowd / unlabelled areas) with the mean colour,
    without touching pixels that belong to a labelled person.
  * writes a dataset YAML and a small statistics report (persons/image, max persons,
    visibility groups) used for the dataset statistics and for the max_det setting.

Usage (examples)
  python prepare_data.py crowdhuman  --root /data/CrowdHuman  --out /work/ch_yolo  --paint-ignore
  python prepare_data.py citypersons --root /data/citypersons --out /work/cp_yolo  --paint-ignore

Expected input layout
  CrowdHuman : root/annotation_train.odgt, root/annotation_val.odgt, root/Images/, root/Images_val/
  CityPersons: root/leftImg8bit/{train,val}/<city>/*_leftImg8bit.png
               root/gtBboxCityPersons/{train,val}/<city>/*_gtBboxCityPersons.json
"""
import argparse, json, os, shutil, sys
from pathlib import Path
import numpy as np
from PIL import Image

NBINS = 20                      # visibility resolution 0.05
IMG_MEAN = (114, 114, 114)      # Ultralytics padding colour, used for ignore regions


def clip_box(b, W, H):
    x, y, w, h = b
    x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    return (x1, y1, max(0, x2 - x1), max(0, y2 - y1))


def vis_ratio(fbox, vbox, W=None, H=None):
    """Eq. (1), occlusion-only: |B_vis ∩ B_full ∩ image| / |B_full ∩ image|.
    Clipping the full box to the image first means that a person cut off by the image border
    (truncation) is NOT counted as occluded — only parts hidden by other objects lower v."""
    if W is not None:
        fbox, vbox = clip_box(fbox, W, H), clip_box(vbox, W, H)
    fx, fy, fw, fh = fbox
    vx, vy, vw, vh = vbox
    iw = max(0.0, min(fx + fw, vx + vw) - max(fx, vx))
    ih = max(0.0, min(fy + fh, vy + vh) - max(fy, vy))
    return float(np.clip(iw * ih / max(fw * fh, 1e-6), 0.0, 1.0))


def vbin(v):
    return int(min(NBINS - 1, max(0, np.floor(v * NBINS))))


def yolo_line(b, vb, W, H, v, hb=None):
    """Pose-format line: full box (clipped, normalised) + 4 keypoints:
    visible-box corners (top-left, bottom-right) and head-box corners (top-left, bottom-right).
    A missing / ignored head gets zeros with visibility flag 0."""
    x1, y1, w, h = clip_box(b, W, H)
    if w < 2 or h < 2:
        return None
    vx1, vy1, vw, vh = clip_box(vb, W, H)
    if vw < 1 or vh < 1:                       # degenerate visible box: fall back to the full box
        vx1, vy1, vw, vh = x1, y1, w, h
    head = "0 0 0 0 0 0"
    if hb is not None:
        hx1, hy1, hw, hh = clip_box(hb, W, H)
        if hw >= 1 and hh >= 1:
            head = f"{hx1/W:.6f} {hy1/H:.6f} 2 {(hx1+hw)/W:.6f} {(hy1+hh)/H:.6f} 2"
    return (f"{vbin(v)} {(x1+w/2)/W:.6f} {(y1+h/2)/H:.6f} {w/W:.6f} {h/H:.6f} "
            f"{vx1/W:.6f} {vy1/H:.6f} 2 {(vx1+vw)/W:.6f} {(vy1+vh)/H:.6f} 2 {head}")


def paint_ignore(img_path, out_path, ignore_boxes, keep_boxes):
    im = np.array(Image.open(img_path).convert("RGB"))
    H, W = im.shape[:2]
    mask = np.zeros((H, W), bool)
    for x, y, w, h in ignore_boxes:
        mask[max(0, int(y)):min(H, int(y + h)), max(0, int(x)):min(W, int(x + w))] = True
    for x, y, w, h in keep_boxes:            # never paint over a labelled person
        mask[max(0, int(y)):min(H, int(y + h)), max(0, int(x)):min(W, int(x + w))] = False
    im[mask] = IMG_MEAN
    Image.fromarray(im).save(out_path, quality=95)
    return float(mask.mean())


def write_image(src, dst, ign, keep, paint):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if paint and ign:
        return paint_ignore(src, dst, ign, keep)
    if not dst.exists():
        try:
            os.symlink(os.path.abspath(src), dst)
        except OSError:
            shutil.copy(src, dst)
    return 0.0


def records_crowdhuman(root, split):
    ann = root / f"annotation_{split}.odgt"
    img_dir = root / ("Images" if split == "train" else "Images_val")
    for line in open(ann):
        r = json.loads(line)
        yield r["ID"], img_dir / f"{r['ID']}.jpg", r["gtboxes"]


def parse_crowdhuman(gtboxes):
    persons, ignore = [], []
    for b in gtboxes:
        if b["tag"] != "person" or b.get("extra", {}).get("ignore", 0) == 1:
            ignore.append(b["fbox"])
        else:
            hb = b.get("hbox")
            if b.get("head_attr", {}).get("ignore", 0) == 1:
                hb = None
            persons.append((b["fbox"], b["vbox"], hb))
    return persons, ignore


def records_citypersons(root, split):
    for jf in sorted((root / "gtBboxCityPersons" / split).glob("*/*_gtBboxCityPersons.json")):
        stem = jf.name.replace("_gtBboxCityPersons.json", "")
        img = root / "leftImg8bit" / split / jf.parent.name / f"{stem}_leftImg8bit.png"
        yield stem, img, json.load(open(jf))


def parse_citypersons(j):
    """Standard protocol: 'pedestrian' is positive; every other label is treated as ignore."""
    persons, ignore = [], []
    for o in j["objects"]:
        if o["label"] == "pedestrian":
            persons.append((o["bbox"], o["bboxVis"], None))   # CityPersons has no head boxes
        else:
            ignore.append(o["bbox"])
    return persons, ignore


def convert(dataset, root, out, paint, only_existing):
    root, out = Path(root), Path(out)
    stats = {}
    for split in ["train", "val"]:
        recs = records_crowdhuman(root, split) if dataset == "crowdhuman" else records_citypersons(root, split)
        n_img, counts, vis, painted, skipped, heads = 0, [], [], [], 0, []
        for stem, img, ann in recs:
            persons, ignore = parse_crowdhuman(ann) if dataset == "crowdhuman" else parse_citypersons(ann)
            counts.append(len(persons))
            if not img.exists():
                skipped += 1
                if only_existing:
                    continue
                raise FileNotFoundError(img)
            W, H = Image.open(img).size
            vs = [vis_ratio(b, vb, W, H) for b, vb, _ in persons]
            vis += vs
            heads += [hb is not None for _, _, hb in persons]
            lines = [l for l in (yolo_line(b, vb, W, H, v, hb) for (b, vb, hb), v in zip(persons, vs)) if l]
            (out / "labels" / split).mkdir(parents=True, exist_ok=True)
            (out / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
            ext = img.suffix if not paint else ".jpg"
            painted.append(write_image(img, out / "images" / split / f"{stem}{ext}", ignore,
                                       [b for b, _, _ in persons], paint))
            n_img += 1
        counts, vis = np.array(counts), np.array(vis)
        stats[split] = dict(
            annotated_images=int(len(counts)), converted_images=n_img, images_missing=skipped,
            persons=int(counts.sum()), persons_per_image=round(float(counts.mean()), 2),
            max_persons_in_one_image=int(counts.max()),
            images_over_300_persons=int((counts > 300).sum()),
            visibility_stats_over_converted_images_only=True,
            visibility_light_ge_0_65=round(float((vis >= .65).mean()), 3),
            visibility_heavy_0_3_to_0_65=round(float(((vis >= .3) & (vis < .65)).mean()), 3),
            visibility_severe_lt_0_3=round(float((vis < .3).mean()), 3),
            persons_with_head_box=round(float(np.mean(heads)) if heads else 0.0, 3),
            mean_fraction_of_image_painted=round(float(np.mean(painted)) if painted else 0.0, 4))
    maxp = max(s["max_persons_in_one_image"] for s in stats.values())
    stats["recommended_max_det"] = int(np.ceil(maxp * 1.3 / 100) * 100)
    yaml = (f"path: {out.resolve()}\ntrain: images/train\nval: images/val\n"
            f"# class field in label files = visibility bin (0..{NBINS-1}); trainer maps it to class 0\n"
            f"nc: 1\nnames: ['person']\nvis_bins: {NBINS}\n"
            f"kpt_shape: [4, 3]   # visible-box corners, then head-box corners (top-left, bottom-right)\n"
            f"flip_idx: [0, 1, 2, 3]\n")
    (out / f"{dataset}.yaml").write_text(yaml)
    (out / "stats.json").write_text(json.dumps(stats, indent=2))
    return stats


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("dataset", choices=["crowdhuman", "citypersons"])
    ap.add_argument("--root", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--paint-ignore", action="store_true", help="fill ignore regions with the mean colour")
    ap.add_argument("--only-existing", action="store_true", help="skip annotations whose image is missing (for samples)")
    a = ap.parse_args()
    print(json.dumps(convert(a.dataset, a.root, a.out, a.paint_ignore, a.only_existing), indent=2))


In [ ]:
%%writefile vgyolo.py
"""
vgyolo.py — Visibility-Guided YOLO (VG-YOLO) on top of Ultralytics (tested with ultralytics 8.4.x).

Adds to any Ultralytics detection model (YOLO26-P2, YOLOv8-P2, ...), without changing backbone/neck/head:
  * VPB   — Visibility Prediction Branch on the stride-4 neck feature N2      (feature-level visibility variant)
  * VGFFM — Visibility-Guided Feature Fusion Module before the detection head (feature-level visibility variant)
  * a joint loss  L = L_det + lambda_vis * L_vis                            

Labels come from prepare_data.py (YOLO pose format: class field = visibility bin, two "keypoints" =
visible-box corners). The trainer turns the class field into (class 0, visibility v) after augmentation.

Modes (feature-level visibility variants, pilot protocol):
  "baseline" : no extra modules (still reads the same labels, so data handling is identical)
  "vpb"      : VPB + visibility loss only, fusion unchanged                  (variant B)
  "asff"     : same fusion module, no visibility input, no prior, no VPB    (variant C)
  "vis_off"  : VGFFM + VPB, but V is replaced by 1 everywhere inside VGFFM  (variant D)
  "no_prior" : VGFFM + VPB with gamma fixed to 0                            (variant E)
  "vg"       : full model                                                   (variant F)
Oracle (variant G) is an evaluation-time option: set_oracle(model, V_star) before a forward pass.
  "vg_oracle": same network as "vg", but during TRAINING the fusion receives the ground-truth map V* instead of
               the prediction (the VPB is still trained with L_vis). Upper-bound diagnostic; evaluate it with
               export_eval.py --oracle.
  "vg_sched" : same network as "vg"; training starts with the ground-truth V* and hands over gradually to the
               model's own prediction (scheduled teacher forcing): each image uses V* with probability p, where
               p = 1 for the first 20% of epochs, falls linearly to 0 at 70%, and stays 0 afterwards.
               At test time only the prediction is used, so its numbers are fair.
"""
from __future__ import annotations

import math
from copy import copy, deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import get_split_fraction  # noqa: F401  (kept for API parity)
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.nn.modules import Conv, Detect
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import colorstr
from ultralytics.utils.torch_utils import unwrap_model
from ultralytics.utils.loss import E2ELoss, v8DetectionLoss
from ultralytics.utils.tal import make_anchors, TaskAlignedAssigner
from functools import partial

# ----------------------------------------------------------------------------- configuration
VG_CFG = dict(
    mode="vg",          # baseline | vpb | asff | vis_off | no_prior | vg
    vis_bins=20,        # must match prepare_data.NBINS
    lambda_vis=1.0,     # Eq. (14)
    lambda_bg=0.1,      # Eq. (3)
    r=8,                # descriptor channels, Eq. (8)
    cm=64,              # common width of the aligned candidates (keeps VGFFM light)
    c_vpb=32,           # hidden width of the VPB
    rho=(1.0, 1 / 3, -1 / 3, -1.0),  # level prior, Eq. (9), shallow -> deep
    gamma_init=1.0,
    vis_weight=0.0,     # Option B: occlusion-aware loss weighting, beta in w = 1 + beta * (1 - v); 0 = off
    vis_weight_norm=True,
    vra="none",
    crm=False,          # Crowd Relation Module on the one-to-one (NMS-free) branch
    vba=False,          # Visible-Body Attention on the neck outputs (feeds BOTH heads)
    vba_r=4, lambda_vba=1.0,
    hg=False,           # Head-Guided module (head-centre heatmap on P2 guiding the one-to-one branch)
    hg_dim=64, lambda_head=1.0,
    crm_dim=64, crm_heads=4, crm_win=7,         # visible-region assignment: none | o2o (one-to-one branch only) | all (both branches)  # rescale the weights of positives to mean 1 (False: visible people keep weight 1)
)


# ----------------------------------------------------------------------------- modules
class VPB(nn.Module):
    """Visibility Prediction Branch, Eq. (4): two 3x3 Conv-BN-SiLU + 1x1 conv -> logits (sigmoid gives V)."""

    def __init__(self, c: int, ch: int = 32):
        super().__init__()
        self.body = nn.Sequential(Conv(c, ch, 3), Conv(ch, ch, 3))
        self.out = nn.Conv2d(ch, 1, 1)
        nn.init.constant_(self.out.bias, 2.0)  # start near V = 0.88 (most pixels are background, target 1)

    def forward(self, x):
        return self.out(self.body(x))


class LevelFusion(nn.Module):
    """VGFFM for one output level l (Eqs. 7–11)."""

    def __init__(self, chs, l, r, use_vis, use_prior, rho, gamma_init, cm=64):
        super().__init__()
        self.l, self.n = l, len(chs)
        cl = chs[l]
        # every level (including l itself) is projected to a common width cm (Eq. 7)
        self.align = nn.ModuleList([Conv(chs[k], cm, 1) for k in range(self.n)])
        self.desc = nn.ModuleList([nn.Conv2d(cm, r, 1) for _ in range(self.n)])
        self.use_vis, self.use_prior = use_vis, use_prior
        self.g = nn.Sequential(nn.Conv2d(self.n * r + int(use_vis), 16, 1), nn.SiLU(), nn.Conv2d(16, self.n, 1))
        # E: depth-wise 3x3 on the fused map, then a 1x1 back to c_l (Eq. 11)
        self.refine = nn.Sequential(Conv(cm, cm, 3, g=cm), Conv(cm, cl, 1, act=False))
        nn.init.zeros_(self.refine[1].bn.weight)  # E starts at 0  ->  F_l = N_l at initialisation
        self.register_buffer("rho", torch.tensor(rho[: self.n], dtype=torch.float32).view(1, -1, 1, 1))
        # gamma = softplus(gamma_bias) >= 0, initialised to gamma_init ("bias" in the name keeps it out of weight decay)
        self.gamma_bias = nn.Parameter(torch.tensor(math.log(math.expm1(gamma_init)) if gamma_init > 0 else -20.0))

    def gamma(self):
        return F.softplus(self.gamma_bias)

    def forward(self, xs, V_l):
        l = self.l
        h, w = xs[l].shape[-2:]
        cands = []
        for k, x in enumerate(xs):
            if k == l:
                c = self.align[k](x)
            elif k > l:  # deeper, smaller map: 1x1 conv then nearest upsample (the two commute)
                c = F.interpolate(self.align[k](x), size=(h, w), mode="nearest")
            else:  # shallower, larger map: max-pool down, then 1x1 conv
                c = self.align[k](F.adaptive_max_pool2d(x, (h, w)))
            cands.append(c)
        z = [self.desc[k](c) for k, c in enumerate(cands)]
        if self.use_vis:
            z.append(V_l)
        logits = self.g(torch.cat(z, 1))                                   # Eq. (8)
        if self.use_prior and V_l is not None:
            logits = logits + self.gamma() * (1.0 - V_l) * self.rho       # Eq. (9) numerator term
        alpha = logits.softmax(1)                                          # Eq. (9)
        fused = sum(alpha[:, k : k + 1] * cands[k] for k in range(self.n))  # Eq. (10)
        self.last_alpha = alpha.detach()
        return xs[l] + self.refine(fused)                                  # Eq. (11)


class VGNeck(nn.Module):
    """Holds VPB + one LevelFusion per detection level. Inputs are the neck outputs fed to Detect (shallow -> deep)."""

    def __init__(self, chs, mode, cfg):
        super().__init__()
        self.mode = mode
        self.use_vpb = mode in {"vpb", "vis_off", "no_prior", "vg"}
        self.use_fusion = mode in {"asff", "vis_off", "no_prior", "vg"}
        use_vis = mode in {"vis_off", "no_prior", "vg"}
        use_prior = mode in {"vis_off", "vg"}
        self.vpb = VPB(chs[0], cfg["c_vpb"]) if self.use_vpb else None
        self.fuse = (
            nn.ModuleList(
                [LevelFusion(chs, l, cfg["r"], use_vis, use_prior, cfg["rho"], cfg["gamma_init"], cfg["cm"])
                 for l in range(len(chs))]
            )
            if self.use_fusion
            else None
        )
        self.cache = {}
        self.oracle = None  # optional ground-truth V* for the oracle experiment (evaluation only)

    def forward(self, xs):
        xs = list(xs)
        V = None
        if self.use_vpb:
            logit = self.vpb(xs[0])
            self.cache["logit"] = logit
            V = torch.sigmoid(logit)
            if self.oracle is not None:
                mask = getattr(self, "oracle_mask", None)   # per-image choice (scheduled hand-over), else all images
                V = self.oracle.to(V) if mask is None else torch.where(mask.to(V.device), self.oracle.to(V), V)
            self.cache["V"] = V.detach()
        if self.use_fusion:
            outs = []
            for l, f in enumerate(self.fuse):
                if f.use_vis:
                    Vl = F.adaptive_avg_pool2d(V, xs[l].shape[-2:])  # Eq. (5): average pooling
                    if self.mode == "vis_off":
                        Vl = torch.ones_like(Vl)
                else:
                    Vl = None
                outs.append(f(xs, Vl))
            xs = outs
        return xs


class CrowdRelationModule(nn.Module):
    """Crowd Relation Module (CRM) for one level of the one-to-one (NMS-free) branch.

    Each location compares itself with its neighbours inside a local k x k window (multi-head self-attention with a
    learned relative-position bias), so candidates of the same person and of different, adjacent persons can be told
    apart before the one-to-one classification/regression. Reduce (1x1) -> window attention -> depth-wise conv FFN ->
    expand (1x1, zero-initialised, so the block starts as the identity) -> residual."""

    def __init__(self, c, d=64, heads=4, win=7):
        super().__init__()
        self.d, self.h, self.win = d, heads, win
        self.reduce = Conv(c, d, 1)
        self.qkv = nn.Conv2d(d, 3 * d, 1)
        self.proj = nn.Conv2d(d, d, 1)
        self.rpb = nn.Parameter(torch.zeros(heads, (2 * win - 1) ** 2))       # relative position bias
        nn.init.trunc_normal_(self.rpb, std=0.02)
        yy, xx = torch.meshgrid(torch.arange(win), torch.arange(win), indexing="ij")
        pts = torch.stack((yy.flatten(), xx.flatten()), 1)
        rel = pts[:, None] - pts[None] + win - 1
        self.register_buffer("rp_index", rel[..., 0] * (2 * win - 1) + rel[..., 1], persistent=False)
        self.ffn = nn.Sequential(Conv(d, d, 3, g=d), Conv(d, d, 1))
        self.expand = Conv(d, c, 1, act=False)
        nn.init.zeros_(self.expand.bn.weight)

    def attn(self, y):
        B, C, H, W = y.shape
        k = self.win
        ph, pw = (k - H % k) % k, (k - W % k) % k
        yp = F.pad(y, (0, pw, 0, ph))
        Hp, Wp = H + ph, W + pw
        q, kk, v = self.qkv(yp).chunk(3, 1)
        def win_split(t):  # (B,C,Hp,Wp) -> (B*nW, heads, k*k, C/heads)
            t = t.view(B, self.h, C // self.h, Hp // k, k, Wp // k, k)
            return t.permute(0, 3, 5, 1, 4, 6, 2).reshape(-1, self.h, k * k, C // self.h)
        q, kk, v = win_split(q), win_split(kk), win_split(v)
        a = (q * (C // self.h) ** -0.5) @ kk.transpose(-2, -1) + self.rpb[:, self.rp_index]
        # padded positions must not be attended to
        if ph or pw:
            m = F.pad(torch.zeros(1, 1, H, W, device=y.device), (0, pw, 0, ph), value=1.0)
            m = m.view(1, Hp // k, k, Wp // k, k).permute(0, 1, 3, 2, 4).reshape(-1, 1, 1, k * k)
            a = a.masked_fill(m.repeat(B, 1, 1, 1).bool(), float("-inf"))
        o = a.softmax(-1) @ v                                                  # (B*nW, heads, k*k, C/heads)
        o = o.view(B, Hp // k, Wp // k, self.h, k, k, C // self.h).permute(0, 3, 6, 1, 4, 2, 5)
        o = o.reshape(B, C, Hp, Wp)[..., :H, :W]
        return self.proj(o)

    def forward(self, x):
        y = self.reduce(x)
        y = y + self.attn(y)
        y = y + self.ffn(y)
        return x + self.expand(y)


class VisibleBodyAttention(nn.Module):
    """Visible-Body Attention (VBA) on every neck output level (P2-P5), before BOTH detection heads.

    A light branch per level (1x1 reduce -> depth-wise 3x3 -> 1x1) predicts a visible-body map M_l, supervised by the
    union of the visible-body boxes. The features are re-weighted as  F'_l = F_l * (1 + g_l * (2 M_l - 1)),
    boosting visible body parts and damping regions where a person is hidden behind someone else. The gate g_l is a
    learnable scalar initialised to 0, so the block starts as the identity while the map is already being learned
    from its own loss."""

    def __init__(self, chs, r=4):
        super().__init__()
        self.branch = nn.ModuleList(
            nn.Sequential(Conv(c, max(c // r, 16), 1), Conv(max(c // r, 16), max(c // r, 16), 3, g=max(c // r, 16)),
                          nn.Conv2d(max(c // r, 16), 1, 1)) for c in chs)
        self.gate_bias = nn.Parameter(torch.zeros(len(chs)))     # "bias" in the name keeps it out of weight decay
        self.cache = {}

    def forward(self, xs):
        logits = [b(x) for b, x in zip(self.branch, xs)]
        self.cache["logits"] = logits
        return [x * (1 + self.gate_bias[i] * (2 * torch.sigmoid(lg) - 1)) for i, (x, lg) in enumerate(zip(xs, logits))]


@torch.no_grad()
def visible_body_target(batch, shape, lambda_bg=0.1):
    """Binary visible-body target (union of visible boxes) and weight map (1 inside full boxes, lambda_bg elsewhere)
    at the resolution of `shape` (B,1,h,w). Visible box = keypoints 1-2 (min/max, flip-safe)."""
    B, _, h, w = shape
    dev = batch["img"].device
    T = torch.zeros((B, 1, h, w), device=dev)
    Wm = torch.full((B, 1, h, w), float(lambda_bg), device=dev)
    if batch["bboxes"].numel() == 0:
        return T, Wm
    xywh = batch["bboxes"].to(dev).float()
    fx1, fy1, fx2, fy2 = xywh[:, 0] - xywh[:, 2] / 2, xywh[:, 1] - xywh[:, 3] / 2, xywh[:, 0] + xywh[:, 2] / 2, xywh[:, 1] + xywh[:, 3] / 2
    kp = batch.get("keypoints")
    if kp is not None and kp.numel():
        kp = kp[:, :2, :2].to(dev).float()
        vx1, vx2 = torch.maximum(kp[:, :, 0].min(1).values, fx1), torch.minimum(kp[:, :, 0].max(1).values, fx2)
        vy1, vy2 = torch.maximum(kp[:, :, 1].min(1).values, fy1), torch.minimum(kp[:, :, 1].max(1).values, fy2)
        bad = (vx2 - vx1 < 1e-4) | (vy2 - vy1 < 1e-4)
        vx1, vy1, vx2, vy2 = [torch.where(bad, a, b) for a, b in ((fx1, vx1), (fy1, vy1), (fx2, vx2), (fy2, vy2))]
    else:
        vx1, vy1, vx2, vy2 = fx1, fy1, fx2, fy2
    gy = (torch.arange(h, device=dev).float() + 0.5) / h
    gx = (torch.arange(w, device=dev).float() + 0.5) / w
    bi = batch["batch_idx"].to(dev).long()
    for b in range(B):
        idx = (bi == b).nonzero().flatten()
        if idx.numel() == 0:
            continue
        inx = lambda a1, a2, g: (g[None] >= a1[:, None]) & (g[None] < a2[:, None])  # noqa: E731
        vis = (inx(vy1[idx], vy2[idx], gy)[:, :, None] & inx(vx1[idx], vx2[idx], gx)[:, None, :]).any(0)
        full = (inx(fy1[idx], fy2[idx], gy)[:, :, None] & inx(fx1[idx], fx2[idx], gx)[:, None, :]).any(0)
        T[b, 0] = vis.float()
        Wm[b, 0] = torch.where(full | vis, torch.ones_like(Wm[b, 0]), Wm[b, 0])
    return T, Wm


class HeadGuide(nn.Module):
    """Head-Guided module (HG).

    A light branch on the finest level (P2) predicts a head-centre heatmap (one peak per person, supervised by the
    CrowdHuman head boxes). Its features and heatmap are pooled to every detection level and added, through a
    zero-initialised 1x1 projection, to the features entering the one-to-one (NMS-free) branch. Heads are rarely
    occluded and rarely overlap, so the one-to-one branch receives an explicit cue of how many people are present
    in a crowded region. The heatmap branch is trained by its own loss; its outputs are detached before fusion, so
    the one-to-one loss trains only the fusion projections."""

    def __init__(self, chs, d=64):
        super().__init__()
        self.stem = nn.Sequential(Conv(chs[0], d, 3), Conv(d, d, 3))
        self.hm = nn.Conv2d(d, 1, 1)
        nn.init.constant_(self.hm.bias, -2.19)                                   # prior 0.1
        self.fuse = nn.ModuleList(Conv(d + 1, c, 1, act=False) for c in chs)
        for f in self.fuse:
            nn.init.zeros_(f.bn.weight)                                           # starts as identity
        self.cache = {}

    def forward(self, x, xo):
        f = self.stem(x[0])
        logit = self.hm(f)
        self.cache["logit"] = logit
        g = torch.cat((f, logit.sigmoid()), 1)
        if self.training:
            g = g.detach()
        out = []
        for l, (xi, fu) in enumerate(zip(xo, self.fuse)):
            gl = g if l == 0 else F.max_pool2d(g, 2 ** l, 2 ** l, ceil_mode=True)[..., : xi.shape[-2], : xi.shape[-1]]
            out.append(xi + fu(gl))
        return out


class VGDetect(Detect):
    """Detect head with optional VGNeck on its inputs and optional CRM in front of the one-to-one branch."""

    def forward(self, x):
        if hasattr(self, "vg"):
            x = self.vg(x)
        if hasattr(self, "vba"):
            x = self.vba(x)
        if not (hasattr(self, "crm") or hasattr(self, "hg")) or getattr(self, "one2one_cv2", None) is None:
            return super().forward(x)
        preds = self.forward_head(x, **self.one2many)
        xo = [xi.detach() for xi in x] if self.training else list(x)
        if hasattr(self, "hg"):
            xo = self.hg(x, xo)
        if hasattr(self, "crm"):
            xo = [m(xi) for m, xi in zip(self.crm, xo)]
        one2one = self.forward_head(xo, **self.one2one)
        preds = {"one2many": preds, "one2one": one2one}
        if self.training:
            return preds
        y = self._inference(preds["one2one" if self.end2end else "one2many"])
        if self.end2end:
            y = self.postprocess(y.permute(0, 2, 1))
        return y if self.export else (y, preds)


class VGLoss:
    """Wraps the base detection loss and adds the visibility loss (Eqs. 6 and 14)."""

    def __init__(self, model, base, cfg):
        self.model, self.base, self.cfg = model, base, cfg

    def __getattr__(self, k):  # pass 'updates', 'update', 'hyp', ... through to the base loss
        if k.startswith("__"):
            raise AttributeError(k)
        return getattr(object.__getattribute__(self, "base"), k)

    def __setattr__(self, k, v):
        if k in {"model", "base", "cfg"}:
            object.__setattr__(self, k, v)
        else:
            setattr(self.base, k, v)

    def __call__(self, preds, batch):
        total, items = self.base(preds, batch)
        det = self.model.model[-1]
        bs = batch["img"].shape[0]
        vg = getattr(det, "vg", None)
        if vg is not None and vg.use_vpb and "logit" in vg.cache and "vis" in batch:
            logit = vg.cache["logit"].float()
            target, weight = visibility_target(batch, logit.shape, self.cfg)
            bce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
            lv = (bce * weight).sum() / weight.sum().clamp(min=1.0)
            items = dict(items)
            items["vis_loss"] = lv.detach()
            total = total + self.cfg["lambda_vis"] * lv * bs
        vba = getattr(det, "vba", None)
        if vba is not None and "logits" in vba.cache:
            lv, n = 0.0, 0
            for lg in vba.cache["logits"]:
                lg = lg.float()
                T, Wm = visible_body_target(batch, lg.shape, self.cfg["lambda_bg"])
                bce = F.binary_cross_entropy_with_logits(lg, T, reduction="none")
                lv = lv + (bce * Wm).sum() / Wm.sum().clamp(min=1.0)
                n += 1
            lv = lv / max(n, 1)
            items = dict(items)
            items["vba_loss"] = lv.detach()
            total = total + self.cfg["lambda_vba"] * lv * bs
        hg = getattr(det, "hg", None)
        if hg is not None and "logit" in hg.cache:
            logit = hg.cache["logit"].float()
            target = head_heatmap_target(batch, logit.shape)
            lh = heatmap_focal_loss(logit, target)
            items = dict(items)
            items["head_loss"] = lh.detach()
            total = total + self.cfg["lambda_head"] * lh * bs
        return total, items


@torch.no_grad()
def head_heatmap_target(batch, shape):
    """Gaussian head-centre heatmap (CenterNet style) at the resolution of `shape` (B,1,h,w).
    Head boxes are keypoints 3 and 4 (normalised corners, visibility flag > 0)."""
    B, _, h, w = shape
    kp = batch.get("keypoints")
    dev = batch["img"].device
    T = torch.zeros((B, 1, h, w), device=dev)
    if kp is None or kp.numel() == 0 or kp.shape[1] < 4:
        return T
    kp = kp.to(dev).float()
    ok = (kp[:, 2, 2] > 0) | (kp[:, 3, 2] > 0)          # keep heads cut by the image border (one corner inside)
    # corners may swap under horizontal flip, so take min / max
    x1, x2 = torch.minimum(kp[:, 2, 0], kp[:, 3, 0]) * w, torch.maximum(kp[:, 2, 0], kp[:, 3, 0]) * w
    y1, y2 = torch.minimum(kp[:, 2, 1], kp[:, 3, 1]) * h, torch.maximum(kp[:, 2, 1], kp[:, 3, 1]) * h
    ok &= (x2 - x1 > 0) & (y2 - y1 > 0) & (x2 > 0) & (y2 > 0) & (x1 < w) & (y1 < h)
    cx, cy = ((x1 + x2) / 2).clamp(0, w - 1e-3), ((y1 + y2) / 2).clamp(0, h - 1e-3)
    sig = (torch.sqrt((x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)) / 6).clamp(min=0.8)
    bi = batch["batch_idx"].to(dev).long()
    gy = torch.arange(h, device=dev).float() + 0.5
    gx = torch.arange(w, device=dev).float() + 0.5
    for b in range(B):
        idx = ((bi == b) & ok).nonzero().flatten()
        if idx.numel() == 0:
            continue
        g = torch.exp(-((gx[None, None, :] - cx[idx, None, None]) ** 2 + (gy[None, :, None] - cy[idx, None, None]) ** 2)
                      / (2 * sig[idx, None, None] ** 2))                                     # (n,h,w)
        T[b, 0] = g.amax(0)
        T[b, 0, cy[idx].long(), cx[idx].long()] = 1.0                                           # exact peaks
    return T


def heatmap_focal_loss(logit, target, alpha=2.0, beta=4.0):
    """Penalty-reduced focal loss of CenterNet, normalised by the number of heads."""
    p = logit.sigmoid().clamp(1e-4, 1 - 1e-4)
    pos = target.eq(1).float()
    pos_loss = torch.log(p) * (1 - p) ** alpha * pos
    neg_loss = torch.log(1 - p) * p ** alpha * (1 - target) ** beta * (1 - pos)
    n = pos.sum().clamp(min=1.0)
    return -(pos_loss.sum() + neg_loss.sum()) / n


class VisibleRegionAssigner(TaskAlignedAssigner):
    """Task-aligned assigner whose candidate anchors must lie inside the person's VISIBLE box (not the full box).
    The alignment metric and the regression target still use the full box. Set `cand_boxes` before each call."""
    cand_boxes = None

    def select_candidates_in_gts(self, xy_centers, gt_bboxes, mask_gt, eps=1e-9):
        boxes = self.cand_boxes if self.cand_boxes is not None and self.cand_boxes.shape == gt_bboxes.shape else gt_bboxes
        return super().select_candidates_in_gts(xy_centers, boxes, mask_gt, eps)


class VisWeightedDetLoss(v8DetectionLoss):
    """Option B: occlusion-aware loss weighting.

    Every anchor assigned to a ground-truth person gets the weight w = 1 + beta * (1 - v), where v is that person's
    visibility (0 = fully hidden, 1 = fully visible). The weights of the positive anchors in a batch are rescaled to
    mean 1, so the overall loss scale stays the same as the baseline and only the balance between visible and hidden
    people changes. Applied to the classification and the box losses. Nothing changes at test time."""

    def __init__(self, model, tal_topk=10, tal_topk2=None, beta=1.0, bins=20, norm=True, vra="none"):
        super().__init__(model, tal_topk, tal_topk2)
        self.beta, self.bins, self.norm = beta, bins, norm
        # visible-region assignment on this branch? (the one-to-one branch is the one built with tal_topk2 == 1)
        self.vra = vra == "all" or (vra == "o2o" and tal_topk2 == 1)
        if self.vra:
            self.assigner.__class__ = VisibleRegionAssigner

    def get_assigned_targets_and_loss(self, preds, batch):
        if "vis" not in batch or (self.beta <= 0 and not self.vra):
            return super().get_assigned_targets_and_loss(preds, batch)
        loss = torch.zeros(3, device=self.device)
        pred_distri = preds["boxes"].permute(0, 2, 1).contiguous()
        pred_scores = preds["scores"].permute(0, 2, 1).contiguous()
        anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        v = (batch["vis"].view(-1, 1).float() + 0.5) / self.bins                       # bin centre -> visibility
        kp = batch.get("keypoints")
        if kp is not None and kp.numel():                                              # visible box (normalised xyxy)
            kp = kp[:, :2, :2].float().to(v.device)          # first two keypoints = visible-box corners
            vb = torch.stack((kp[:, :, 0].min(1).values, kp[:, :, 1].min(1).values,
                              kp[:, :, 0].max(1).values, kp[:, :, 1].max(1).values), 1)
        else:
            vb = torch.zeros((v.shape[0], 4), device=v.device)
        targets = torch.cat((batch["batch_idx"].view(-1, 1), batch["cls"].view(-1, 1), batch["bboxes"], v, vb), 1)
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes, gt_vis, gt_vbox = targets.split((1, 4, 1, 4), 2)
        if self.vra:
            vbox = gt_vbox * imgsz[[1, 0, 1, 0]]
            vbox = torch.cat((torch.maximum(vbox[..., :2], gt_bboxes[..., :2]), torch.minimum(vbox[..., 2:], gt_bboxes[..., 2:])), -1)
            bad = ((vbox[..., 2] - vbox[..., 0]) < 1) | ((vbox[..., 3] - vbox[..., 1]) < 1)
            self.assigner.cand_boxes = torch.where(bad.unsqueeze(-1), gt_bboxes, vbox)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
        _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(
            pred_scores.detach().sigmoid(), (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_scores_sum = target_scores.sum().clamp_(min=1)

        # per-anchor weight from the visibility of the assigned person (positives only), mean 1 over positives
        v_anchor = gt_vis.squeeze(-1).gather(1, target_gt_idx.long()) if gt_vis.shape[1] else torch.ones_like(fg_mask, dtype=torch.float)
        w = 1.0 + self.beta * (1.0 - v_anchor.clamp(0, 1))
        fgf = fg_mask.float()
        if self.norm:
            w = w / ((w * fgf).sum() / fgf.sum().clamp(min=1))
        w = torch.where(fg_mask, w, torch.ones_like(w))

        bce_loss = self.bce(pred_scores, target_scores.to(dtype)) * w.unsqueeze(-1).to(dtype)
        loss[1] = bce_loss.sum() / target_scores_sum
        loss[0], loss[2] = self.bbox_loss(pred_distri, pred_bboxes, anchor_points, target_bboxes / stride_tensor,
                                          target_scores * w.unsqueeze(-1), target_scores_sum, fg_mask, imgsz,
                                          stride_tensor)
        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl
        return ((fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor), loss,
                dict(zip(self.loss_names, loss.detach())))


class VGDetectionModel(DetectionModel):
    """DetectionModel whose criterion includes the visibility loss."""

    def init_criterion(self):
        beta, vra = VG_CFG.get("vis_weight", 0.0), VG_CFG.get("vra", "none")
        if beta > 0 or vra != "none":
            fn = partial(VisWeightedDetLoss, beta=beta, bins=VG_CFG["vis_bins"], norm=VG_CFG.get("vis_weight_norm", True),
                         vra=vra)
            base = E2ELoss(self, loss_fn=fn) if getattr(self.model[-1], "one2one_cv2", None) is not None else fn(self)
        else:
            base = super().init_criterion()
        return VGLoss(self, base, VG_CFG)


# ----------------------------------------------------------------------------- targets
@torch.no_grad()
def visibility_target(batch, shape, cfg):
    """Dense target V* (Eq. 2) and weight map m (Eq. 3) at the resolution of `shape` (B,1,h,w).

    Uses the boxes *after* augmentation: full box from batch['bboxes'] (normalised xywh), visible box from the two
    keypoints (normalised corners), visibility from batch['vis'] (bin index)."""
    B, _, h, w = shape
    dev = batch["bboxes"].device
    V = torch.ones((B, 1, h, w), device=dev)
    M = torch.full((B, 1, h, w), float(cfg["lambda_bg"]), device=dev)
    if batch["bboxes"].numel() == 0:
        return V, M
    gy = (torch.arange(h, device=dev).float() + 0.5) / h
    gx = (torch.arange(w, device=dev).float() + 0.5) / w
    bi = batch["batch_idx"].long()
    xywh = batch["bboxes"].float()
    fx1, fy1 = xywh[:, 0] - xywh[:, 2] / 2, xywh[:, 1] - xywh[:, 3] / 2
    fx2, fy2 = xywh[:, 0] + xywh[:, 2] / 2, xywh[:, 1] + xywh[:, 3] / 2
    v = (batch["vis"].view(-1).float() + 0.5) / cfg["vis_bins"]
    if "keypoints" in batch and batch["keypoints"] is not None and batch["keypoints"].numel():
        kp = batch["keypoints"][:, :2, :2].float()          # first two keypoints = visible-box corners
        vx1, vx2 = kp[:, :, 0].min(1).values, kp[:, :, 0].max(1).values
        vy1, vy2 = kp[:, :, 1].min(1).values, kp[:, :, 1].max(1).values
        # keep the visible box inside the (augmented) full box
        vx1, vy1 = torch.maximum(vx1, fx1), torch.maximum(vy1, fy1)
        vx2, vy2 = torch.minimum(vx2, fx2), torch.minimum(vy2, fy2)
        bad = (vx2 - vx1 < 1e-4) | (vy2 - vy1 < 1e-4)
        vx1, vy1, vx2, vy2 = [torch.where(bad, a, b) for a, b in ((fx1, vx1), (fy1, vy1), (fx2, vx2), (fy2, vy2))]
    else:
        vx1, vy1, vx2, vy2 = fx1, fy1, fx2, fy2
    for b in range(B):
        idx = (bi == b).nonzero().flatten()
        if idx.numel() == 0:
            continue
        inx = lambda a1, a2, g: (g[None] >= a1[:, None]) & (g[None] < a2[:, None])  # noqa: E731
        vis_mask = inx(vy1[idx], vy2[idx], gy)[:, :, None] & inx(vx1[idx], vx2[idx], gx)[:, None, :]  # (n,h,w)
        full_mask = inx(fy1[idx], fy2[idx], gy)[:, :, None] & inx(fx1[idx], fx2[idx], gx)[:, None, :]
        vals = torch.where(vis_mask, v[idx].view(-1, 1, 1), torch.ones_like(vis_mask, dtype=torch.float32))
        V[b, 0] = vals.min(0).values                                          # Eq. (2): min over overlaps
        M[b, 0] = torch.where(full_mask.any(0), torch.ones_like(M[b, 0]), M[b, 0])  # Eq. (3)
    return V, M


def set_oracle(model, V_star):
    """Oracle experiment (variant G): feed ground-truth V* instead of the prediction (evaluation only)."""
    unwrap_model(model).model[-1].vg.oracle = V_star


# ----------------------------------------------------------------------------- model surgery
def attach_vg(model: DetectionModel, mode: str, cfg=VG_CFG):
    """Turn a built Ultralytics DetectionModel into VG-YOLO in place."""
    model.__class__ = VGDetectionModel
    det = model.model[-1]
    assert type(det) in {Detect, VGDetect}, f"expected a Detect head, got {type(det)}"
    det.__class__ = VGDetect
    if mode != "baseline":
        chs = [seq[0].conv.in_channels for seq in det.cv2]
        det.vg = VGNeck(chs, "vg" if mode in {"vg_oracle", "vg_sched"} else mode, cfg).to(next(model.parameters()).device)
    if cfg.get("crm") and getattr(det, "one2one_cv2", None) is not None:
        chs = [seq[0].conv.in_channels for seq in det.cv2]
        det.crm = nn.ModuleList(CrowdRelationModule(c, cfg["crm_dim"], cfg["crm_heads"], cfg["crm_win"]) for c in chs
                                ).to(next(model.parameters()).device)
    if cfg.get("vba"):
        chs = [seq[0].conv.in_channels for seq in det.cv2]
        det.vba = VisibleBodyAttention(chs, cfg["vba_r"]).to(next(model.parameters()).device)
    if cfg.get("hg") and getattr(det, "one2one_cv2", None) is not None:
        chs = [seq[0].conv.in_channels for seq in det.cv2]
        det.hg = HeadGuide(chs, cfg["hg_dim"]).to(next(model.parameters()).device)
    return model


# ----------------------------------------------------------------------------- data
def _vis_data(data, cfg):
    """Dataset view of the data dict: the class field holds visibility bins, so the dataset needs vis_bins names."""
    d = dict(data)
    d["names"] = {i: f"vis{i}" for i in range(cfg["vis_bins"])}
    d["nc"] = cfg["vis_bins"]
    d.setdefault("kpt_shape", [2, 3])
    d.setdefault("flip_idx", [0, 1])
    return d


def build_vg_dataset(args, img_path, batch, data, mode, stride, cfg=VG_CFG):
    a = copy(args)
    a.task = "pose"  # load the two visible-box corners as keypoints
    return YOLODataset(
        img_path=img_path, imgsz=a.imgsz, batch_size=batch, augment=mode == "train", hyp=a,
        rect=a.rect or mode == "val", cache=a.cache or None, single_cls=False, stride=stride,
        pad=0.0 if mode == "train" else 0.5, prefix=colorstr(f"{mode}: "), task="pose", classes=None,
        data=_vis_data(data, cfg), fraction=a.fraction if mode == "train" else 1.0,
    )


def split_vis(batch):
    """Move visibility bins out of the class field; class becomes 0 (single 'person' class)."""
    if "cls" in batch and "vis" not in batch:
        batch["vis"] = batch["cls"].clone()
        batch["cls"] = torch.zeros_like(batch["cls"])
    return batch


def sched_p(epoch, epochs, start=0.2, end=0.7):
    """Probability of feeding ground-truth V* at a given (0-based) epoch for mode vg_sched."""
    f = epoch / max(epochs, 1)
    return 1.0 if f < start else 0.0 if f >= end else 1.0 - (f - start) / (end - start)


class VGValidator(DetectionValidator):
    def build_dataset(self, img_path, mode="val", batch=None):
        return build_vg_dataset(self.args, img_path, batch, self.data, mode, stride=self.stride)

    def preprocess(self, batch):
        return split_vis(super().preprocess(batch))


class VGTrainer(DetectionTrainer):
    vg_cfg = VG_CFG

    def get_model(self, cfg=None, weights=None, verbose=True):
        name = str(cfg.get("yaml_file", "") if isinstance(cfg, dict) else cfg)
        if weights is not None and "-p2" in name:  # index-shifted transfer for P2 models (see load_p2_pretrained)
            model = super().get_model(cfg, None, verbose)
            fam = "yolov8" if "yolov8" in name else "yolo26" if "yolo26" in name else None
            if fam:
                n, tot = load_p2_pretrained(model, weights, fam)
                print(f"VG-YOLO: transferred {n}/{tot} items from pretrained weights ({fam} -> P2 mapping)")
            else:
                model.load(weights)
        else:
            model = super().get_model(cfg, weights, verbose)
        return attach_vg(model, self.vg_cfg["mode"], self.vg_cfg)

    def build_dataset(self, img_path, mode="train", batch=None):
        gs = max(int(unwrap_model(self.model).stride.max()), 32)
        return build_vg_dataset(self.args, img_path, batch, self.data, mode, stride=gs, cfg=self.vg_cfg)

    def preprocess_batch(self, batch):
        batch = split_vis(super().preprocess_batch(batch))
        mode = self.vg_cfg["mode"]
        if mode in {"vg_oracle", "vg_sched"}:  # feed ground-truth V* to the fusion for this training step
            neck = unwrap_model(self.model).model[-1].vg
            B, _, H, W = batch["img"].shape
            p = 1.0 if mode == "vg_oracle" else sched_p(self.epoch, self.epochs)
            if p <= 0:
                neck.oracle, neck.oracle_mask = None, None
            else:
                V, _ = visibility_target(batch, (B, 1, H // 4, W // 4), self.vg_cfg)
                neck.oracle = V
                neck.oracle_mask = None if p >= 1 else (torch.rand(B, 1, 1, 1, device=V.device) < p)
        return batch

    def get_validator(self):
        v = VGValidator(self.test_loader, save_dir=self.save_dir, args=copy(self.args), _callbacks=self.callbacks)
        return v

    def plot_training_labels(self):  # class field holds visibility bins; skip the class histogram
        pass


# ----------------------------------------------------------------------------- pretrained weights for P2 models
_P2_LAYER_MAP = {"yolo26": {17: 23, 19: 25, 20: 26, 22: 28, 23: 29}, "yolov8": {16: 22, 18: 24, 19: 25, 21: 27, 22: 28}}


def load_p2_pretrained(model, weights, family="yolo26"):
    """Transfer COCO weights of a 3-level model (e.g. yolo26s.pt) into the 4-level P2 model.

    Ultralytics' default transfer matches layers by index, which breaks after the extra P2 layers are inserted
    (only the backbone and the first neck layers transfer). Here the shifted neck layers are re-indexed and the
    P3–P5 detection branches are moved to head positions 1–3; everything is then copied where shapes agree."""
    if isinstance(weights, (str, bytes)) or hasattr(weights, "__fspath__"):
        ck = torch.load(weights, map_location="cpu", weights_only=False)
        weights = ck.get("ema") or ck["model"]
    src = weights.float().state_dict() if hasattr(weights, "state_dict") else dict(weights)
    lmap = _P2_LAYER_MAP[family]
    det_src = max(lmap)
    out = {}
    for k, v in src.items():
        parts = k.split(".")
        i = int(parts[1])
        if i in lmap:
            parts[1] = str(lmap[i])
            if i == det_src and len(parts) > 3 and parts[2] in {"cv2", "cv3", "one2one_cv2", "one2one_cv3"}:
                parts[3] = str(int(parts[3]) + 1)  # P3,P4,P5 branches -> positions 1,2,3 (0 is the new P2 branch)
        elif i > min(lmap) - 1 and i not in lmap:
            continue  # layers without a counterpart
        out[".".join(parts)] = v
    dst = model.state_dict()
    ok = {k: v for k, v in out.items() if k in dst and dst[k].shape == v.shape}
    model.load_state_dict(ok, strict=False)
    return len(ok), len(dst)


In [ ]:
%%writefile export_eval.py
"""
export_eval.py — run a trained model on the CrowdHuman validation images and score it with the OFFICIAL CrowdDet
evaluation code (AP, MR^-2, JI), so the numbers are comparable with published papers (Section IV-B of the paper).

Usage
  python export_eval.py --weights runs/.../best.pt --images /data/CrowdHuman/Images_val \
      --gt /data/CrowdHuman/annotation_val.odgt --out results/vg_seed0 --imgsz 640 --crowddet /work/CrowdDet

  --crowddet : path to a clone of https://github.com/Purkialo/CrowdDet (only lib/evaluate and lib/utils are used)
  --subset   : optional visibility group (occlusion subsets): light | heavy | severe  (others become ignore)
  --oracle   : oracle experiment (variant G): the model's predicted visibility map is replaced by the
               ground-truth map V* built from the annotation file (same rule as training). VG models only.
               Also reports how close the model's own prediction is to V* (vis_mae_person, vis_mae_occluded).

Writes <out>/dt.odgt (one JSON line per image, CrowdDet format) and <out>/metrics.json.
Detections are made on the original (un-painted) validation images; ignore regions are handled by the evaluator.
"""
import argparse, json, os, subprocess, sys
from pathlib import Path

import numpy as np
from PIL import Image


def vis_ratio(fbox, vbox, W, H):
    def clip(b):
        x, y, w, h = b
        x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
        return x1, y1, max(0, x2 - x1), max(0, y2 - y1)
    fx, fy, fw, fh = clip(fbox); vx, vy, vw, vh = clip(vbox)
    iw = max(0, min(fx + fw, vx + vw) - max(fx, vx)); ih = max(0, min(fy + fh, vy + vh) - max(fy, vy))
    return min(1.0, iw * ih / max(fw * fh, 1e-6))


NBINS = 20  # must match prepare_data.NBINS


def clip_box(b, W, H):
    x, y, w, h = b
    x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    return x1, y1, x2, y2


class Oracle:
    """Forward pre-hook on the VG neck: builds V* for the current image at the P2 resolution and hands it to the
    model in place of the prediction. Letterbox mapping follows Ultralytics' LetterBox (centre padding)."""

    def __init__(self, neck, imgsz):
        self.neck, self.imgsz, self.persons, self.W, self.H = neck, imgsz, [], 1, 1
        self.err_sum = self.err_n = self.occ_sum = self.occ_n = 0.0
        neck.register_forward_pre_hook(self.hook)

    def set_image(self, gtboxes, W, H):
        self.W, self.H, self.persons = W, H, []
        for g in gtboxes:
            if g["tag"] != "person" or g.get("extra", {}).get("ignore", 0) == 1:
                continue
            v = vis_ratio(g["fbox"], g["vbox"], W, H)
            vc = (min(NBINS - 1, max(0, int(np.floor(v * NBINS)))) + 0.5) / NBINS   # bin centre, as in training
            f = clip_box(g["fbox"], W, H); vb = clip_box(g["vbox"], W, H)
            vb = (max(vb[0], f[0]), max(vb[1], f[1]), min(vb[2], f[2]), min(vb[3], f[3]))
            if vb[2] - vb[0] < 1 or vb[3] - vb[1] < 1:
                vb = f
            self.persons.append((f, vb, vc))

    def hook(self, module, args):
        import torch
        x0 = args[0][0]
        h, w = x0.shape[-2:]
        s = 4                                       # P2 stride
        Hin, Win = h * s, w * s
        r = min(self.imgsz / self.H, self.imgsz / self.W)
        nw, nh = round(self.W * r), round(self.H * r)
        left, top = round((Win - nw) / 2 - 0.1), round((Hin - nh) / 2 - 0.1)
        gx = (torch.arange(w, device=x0.device).float() + 0.5) * s
        gy = (torch.arange(h, device=x0.device).float() + 0.5) * s
        V = torch.ones((1, 1, h, w), device=x0.device)
        full = torch.zeros((h, w), dtype=torch.bool, device=x0.device)
        for f, vb, vc in self.persons:
            m = lambda b: ((gy >= b[1] * r + top) & (gy < b[3] * r + top))[:, None] & \
                          ((gx >= b[0] * r + left) & (gx < b[2] * r + left))[None, :]  # noqa: E731
            V[0, 0] = torch.where(m(vb), torch.minimum(V[0, 0], torch.tensor(vc, device=x0.device)), V[0, 0])
            full |= m(f)
        self.full = full
        module.oracle = V.to(x0.dtype)

    def record(self):
        """Compare the model's own prediction (still computed, then replaced) with V* inside person boxes."""
        import torch
        pred = torch.sigmoid(self.neck.cache["logit"].float())[0, 0]
        tgt = self.neck.oracle.float()[0, 0]
        if self.full.any():
            d = (pred - tgt).abs()
            self.err_sum += float(d[self.full].sum()); self.err_n += float(self.full.sum())
            occ = self.full & (tgt < 0.65)
            self.occ_sum += float(d[occ].sum()); self.occ_n += float(occ.sum())


def in_subset(v, subset):
    return {"light": v >= 0.65, "heavy": 0.3 <= v < 0.65, "severe": v < 0.3}.get(subset, True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True); ap.add_argument("--images", required=True)
    ap.add_argument("--gt", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--imgsz", type=int, default=640); ap.add_argument("--max-det", type=int, default=500)
    ap.add_argument("--conf", type=float, default=0.001); ap.add_argument("--device", default=None)
    ap.add_argument("--crowddet", default=None); ap.add_argument("--subset", default=None)
    ap.add_argument("--oracle", action="store_true", help="feed ground-truth visibility V* (VG models only)")
    ap.add_argument("--only-existing", action="store_true", help="evaluate only images present in --images (samples)")
    a = ap.parse_args()

    sys.path.insert(0, str(Path(__file__).parent))
    import vgyolo  # noqa: F401  (registers the VG classes so the checkpoint can be unpickled)
    from ultralytics import YOLO

    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    model = YOLO(a.weights)
    gts = [json.loads(l) for l in open(a.gt)]
    if a.only_existing:
        gts = [r for r in gts if (Path(a.images) / f"{r['ID']}.jpg").exists()]
    oracle = None
    if a.oracle:
        # the predictor works on its own copy of the network: build it once, then hook that copy
        first = next(Path(a.images) / f"{r['ID']}.jpg" for r in gts)
        model.predict(str(first), imgsz=a.imgsz, device=a.device, verbose=False)
        net = model.predictor.model.model
        neck = getattr(net.model[-1], "vg", None)
        assert neck is not None and neck.use_vpb, "--oracle needs a VG model with a visibility branch"
        oracle = Oracle(neck, a.imgsz)
    dt_lines, gt_lines = [], []
    for r in gts:
        img = Path(a.images) / f"{r['ID']}.jpg"
        W, H = Image.open(img).size
        if oracle:
            oracle.set_image(r["gtboxes"], W, H)
        res = model.predict(str(img), imgsz=a.imgsz, conf=a.conf, max_det=a.max_det, device=a.device, verbose=False)[0]
        if oracle:
            oracle.record()
        b = res.boxes
        xyxy = b.xyxy.cpu().numpy() if len(b) else np.zeros((0, 4))
        sc = b.conf.cpu().numpy() if len(b) else np.zeros(0)
        dts = [{"box": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s), "tag": 1}
               for (x1, y1, x2, y2), s in zip(xyxy, sc)]
        gtb, gt_rec = [], dict(r)
        for g in r["gtboxes"]:
            ign = g["tag"] != "person" or g.get("extra", {}).get("ignore", 0) == 1
            if not ign and a.subset:
                ign = not in_subset(vis_ratio(g["fbox"], g["vbox"], W, H), a.subset)
            gtb.append({"box": g["fbox"], "tag": -1 if ign else 1})
            if a.subset and ign and g["tag"] == "person":
                g = dict(g); g["extra"] = dict(g.get("extra", {}), ignore=1)
            gt_rec.setdefault("_g", []).append(g)
        gt_rec["gtboxes"] = gt_rec.pop("_g", [])
        dt_lines.append(json.dumps({"ID": r["ID"], "height": H, "width": W, "dtboxes": dts, "gtboxes": gtb}))
        gt_lines.append(json.dumps(gt_rec))
    (out / "dt.odgt").write_text("\n".join(dt_lines) + "\n")
    (out / "gt_used.odgt").write_text("\n".join(gt_lines) + "\n")
    print(f"wrote {len(dt_lines)} images to {out/'dt.odgt'}")
    if oracle:
        diag = dict(vis_mae_person=oracle.err_sum / max(oracle.err_n, 1), vis_mae_occluded=oracle.occ_sum / max(oracle.occ_n, 1))
        (out / "oracle_diag.json").write_text(json.dumps(diag, indent=2))
        print("visibility guess error (0 = perfect):", {k: round(v, 4) for k, v in diag.items()})

    if a.crowddet:
        lib = Path(a.crowddet) / "lib"
        code = (
            "import json,sys;sys.path.insert(0,'.');"
            "from evaluate import compute_JI, compute_APMR;"
            f"AP,MR=compute_APMR.compute_APMR(r'{(out/'dt.odgt').resolve()}', r'{(out/'gt_used.odgt').resolve()}', 'box');"
            f"_,JI=compute_JI.evaluation_all(r'{(out/'dt.odgt').resolve()}', 'box');"
            f"json.dump(dict(AP=AP,MR=MR,JI=JI),open(r'{(out/'metrics.json').resolve()}','w'),indent=2);"
            "print('AP %.4f  MR %.4f  JI %.4f'%(AP,MR,JI))"
        )
        subprocess.run([sys.executable, "-c", code], cwd=lib, check=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile score_subsets.py
"""
score_subsets.py — re-score an existing dt.odgt (from export_eval.py, no --subset) on the visibility subsets
light / heavy / severe WITHOUT running the model again. Uses the image sizes stored in dt.odgt.

  python score_subsets.py --dt results/run/dt.odgt --gt annotation_val.odgt --out results/run --crowddet CrowdDet
Writes <out>_light/, <out>_heavy/, <out>_severe/ each with dt.odgt, gt_used.odgt, metrics.json.
"""
import argparse, json, subprocess, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent))
from export_eval import vis_ratio, in_subset  # noqa: E402


def score(out, crowddet):
    lib = Path(crowddet) / "lib"
    code = ("import json,sys;sys.path.insert(0,'.');from evaluate import compute_JI, compute_APMR;"
            f"AP,MR=compute_APMR.compute_APMR(r'{(out/'dt.odgt').resolve()}', r'{(out/'gt_used.odgt').resolve()}', 'box');"
            f"_,JI=compute_JI.evaluation_all(r'{(out/'dt.odgt').resolve()}', 'box');"
            f"json.dump(dict(AP=AP,MR=MR,JI=JI),open(r'{(out/'metrics.json').resolve()}','w'),indent=2);"
            "print('AP %.4f  MR %.4f  JI %.4f'%(AP,MR,JI))")
    subprocess.run([sys.executable, "-c", code], cwd=lib, check=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dt", required=True); ap.add_argument("--gt", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--crowddet", required=True)
    ap.add_argument("--subsets", default="light,heavy,severe")
    a = ap.parse_args()
    dts = {json.loads(l)["ID"]: json.loads(l) for l in open(a.dt)}
    gts = [json.loads(l) for l in open(a.gt)]
    gts = [r for r in gts if r["ID"] in dts]
    for subset in a.subsets.split(","):
        out = Path(f"{a.out}_{subset}"); out.mkdir(parents=True, exist_ok=True)
        dt_lines, gt_lines = [], []
        for r in gts:
            d = dts[r["ID"]]; W, H = d["width"], d["height"]
            gtb, gt_rec = [], dict(r)
            for g in r["gtboxes"]:
                ign = g["tag"] != "person" or g.get("extra", {}).get("ignore", 0) == 1
                if not ign:
                    ign = not in_subset(vis_ratio(g["fbox"], g["vbox"], W, H), subset)
                gtb.append({"box": g["fbox"], "tag": -1 if ign else 1})
                if ign and g["tag"] == "person":
                    g = dict(g); g["extra"] = dict(g.get("extra", {}), ignore=1)
                gt_rec.setdefault("_g", []).append(g)
            gt_rec["gtboxes"] = gt_rec.pop("_g", [])
            dt_lines.append(json.dumps({"ID": r["ID"], "height": H, "width": W, "dtboxes": d["dtboxes"], "gtboxes": gtb}))
            gt_lines.append(json.dumps(gt_rec))
        (out / "dt.odgt").write_text("\n".join(dt_lines) + "\n")
        (out / "gt_used.odgt").write_text("\n".join(gt_lines) + "\n")
        print(subset, end=": "); score(out, a.crowddet)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_run.py
"""
train_run.py — one VG-YOLO training run from the command line (used to run two experiments in parallel,
one per GPU, on Kaggle's free "GPU T4 x2").

  python train_run.py --mode vg --base yolo26s-p2 --data /kaggle/working/ch_yolo/crowdhuman.yaml \
      --epochs 60 --seed 0 --name yolo26s-p2_vg_s0 --project /kaggle/working/runs --device 0
  python train_run.py --resume /path/to/last.pt --mode vg      (continue an interrupted run)
"""
import argparse, json, sys, time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
import vgyolo  # noqa: E402
from ultralytics import YOLO  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--mode", required=True, choices=["baseline", "vpb", "asff", "vis_off", "no_prior", "vg", "vg_oracle", "vg_sched"])
ap.add_argument("--base", default="yolo26s-p2")
ap.add_argument("--data", default=None)
ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--fraction", type=float, default=1.0)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--imgsz", type=int, default=640)
ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--workers", type=int, default=2)
ap.add_argument("--device", default="0")
ap.add_argument("--project", default="runs")
ap.add_argument("--name", default=None)
ap.add_argument("--pretrained", default=None, help="default: COCO weights of the base, e.g. yolo26s.pt")
ap.add_argument("--vis-weight", type=float, default=0.0, help="Option B: beta for occlusion-aware loss weighting")
ap.add_argument("--no-vw-norm", action="store_true", help="Option B: do not rescale weights to mean 1")
ap.add_argument("--vra", default="none", choices=["none", "o2o", "all"], help="visible-region assignment")
ap.add_argument("--crm", action="store_true", help="add the Crowd Relation Module to the one-to-one branch")
ap.add_argument("--hg", action="store_true", help="add the Head-Guided module (needs head boxes in the labels)")
ap.add_argument("--vba", action="store_true", help="add Visible-Body Attention to the neck outputs")
ap.add_argument("--max-hours", type=float, default=0.0, help="emergency brake: stop training (keeping best.pt) if it exceeds this many hours; epochs stay fixed")
ap.add_argument("--close-mosaic", type=int, default=10)
ap.add_argument("--resume", default=None, help="path to last.pt of an interrupted run")
a = ap.parse_args()

vgyolo.VG_CFG["mode"] = a.mode
vgyolo.VG_CFG["vis_weight"] = a.vis_weight
vgyolo.VG_CFG["vis_weight_norm"] = not a.no_vw_norm
vgyolo.VG_CFG["vra"] = a.vra
vgyolo.VG_CFG["crm"] = a.crm
vgyolo.VG_CFG["hg"] = a.hg
vgyolo.VG_CFG["vba"] = a.vba
t0 = time.time()
if a.resume:
    m = YOLO(a.resume)
    m.train(trainer=vgyolo.VGTrainer, resume=True)
else:
    name = a.name or f"{a.base}_{a.mode}_s{a.seed}_e{a.epochs}_f{a.fraction}"
    pre = a.pretrained or a.base.split("-")[0] + ".pt"
    m = YOLO(f"{a.base}.yaml")
    if a.max_hours > 0:
        def _brake(trainer):
            if (time.time() - trainer.train_time_start) > a.max_hours * 3600:
                print(f"MAX_HOURS reached ({a.max_hours} h) - stopping after epoch {trainer.epoch + 1}")
                trainer.stop = True
        m.add_callback("on_fit_epoch_end", _brake)
    m.train(
        trainer=vgyolo.VGTrainer, data=a.data, pretrained=pre, epochs=a.epochs, fraction=a.fraction,
        imgsz=a.imgsz, batch=a.batch, seed=a.seed, deterministic=True, max_det=500, workers=a.workers,
        device=a.device, project=a.project, name=name, exist_ok=True, save_period=10, plots=True,
        close_mosaic=a.close_mosaic)
save_dir = Path(m.trainer.save_dir)
hours = (time.time() - t0) / 3600
info = dict(mode=a.mode, max_hours=a.max_hours, epochs_done=int(getattr(m.trainer, "epoch", -1)) + 1, vis_weight=a.vis_weight, vis_weight_norm=not a.no_vw_norm, vra=a.vra, crm=a.crm, hg=a.hg, vba=a.vba, base=a.base, epochs=a.epochs, fraction=a.fraction, seed=a.seed, imgsz=a.imgsz,
            batch=a.batch, train_hours=round(hours, 3), save_dir=str(save_dir))
(save_dir / "run_info.json").write_text(json.dumps(info, indent=2))
print("RUN_INFO", json.dumps(info))


In [ ]:
# Convert CrowdHuman once (~20-40 min) and download the COCO weights once
if not os.path.exists(f"{DATA}/crowdhuman.yaml"):
    !python prepare_data.py crowdhuman --root {CH_ROOT} --out {DATA} --paint-ignore
print(open(f"{DATA}/stats.json").read())
from ultralytics import YOLO
YOLO("yolo26s.pt")   # COCO weights (used by both runs)

In [ ]:
# Build the two job lists (one per GPU) and run them in parallel. Progress is printed every 10 minutes.
import subprocess, time, sys
os.makedirs(f"{WORK}/logs", exist_ok=True)
SUBSETS = ["", "light", "heavy", "severe"]

def train_cmd(mode, seed, vw=0.0, norm=True, vra="none", extra="", base="yolo26s-p2"):
    tag = mode + (f"_vw{vw:g}" if vw else "") + ("" if norm else "nn") + ("" if vra == "none" else f"_vra{vra}")           + "".join("_" + e.strip("-") for e in extra.split())
    name = f"{base}_{tag}_s{seed}_e{EPOCHS}_f{FRACTION}"
    return name, (f"python train_run.py --mode {mode} --vis-weight {vw} --base {base} --data {DATA}/crowdhuman.yaml --epochs {EPOCHS} "
                  f"--fraction {FRACTION} --seed {seed} --imgsz {IMGSZ} --batch {BATCH} --workers 2 --device 0 "
                  f"--project {WORK}/runs --name {name} --max-hours {MAX_HOURS} --close-mosaic {CLOSE_MOSAIC}"
                  + ("" if norm else " --no-vw-norm") + f" --vra {vra} {extra}")

def eval_cmds(weights, name):
    return [f"python export_eval.py --weights {weights} --images {CH_ROOT}/Images_val --gt {CH_ROOT}/annotation_val.odgt "
            f"--out {WORK}/results/{name} --imgsz {IMGSZ} --crowddet {WORK}/CrowdDet --device 0",
            f"python score_subsets.py --dt {WORK}/results/{name}/dt.odgt --gt {CH_ROOT}/annotation_val.odgt "
            f"--out {WORK}/results/{name} --crowddet {WORK}/CrowdDet"]

n0, t0 = train_cmd("baseline", 0, 1.0, base="yolo26s-p2")
n1, t1 = train_cmd("baseline", 0, 1.0, extra="--vba", base="yolo26s-p2")
gpu0 = [t0] + eval_cmds(f"{WORK}/runs/{n0}/weights/best.pt", n0)
gpu1 = [t1] + eval_cmds(f"{WORK}/runs/{n1}/weights/best.pt", n1)
NAMES = [n0, n1]

procs = []
for dev, cmds in [("0", gpu0), ("1", gpu1)]:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=dev)
    log = open(f"{WORK}/logs/gpu{dev}.log", "w")
    procs.append((dev, subprocess.Popen(" && ".join(cmds), shell=True, env=env, stdout=log, stderr=subprocess.STDOUT)))
    print("started GPU", dev)
    time.sleep(60)

def tail(path):
    try:
        t = open(path, errors="ignore").read().replace("\r", "\n").split("\n")
        return " | ".join([l for l in t if l.strip()][-2:])[-250:]
    except Exception as e:
        return str(e)

while any(p.poll() is None for _, p in procs):
    time.sleep(600)
    for dev, p in procs:
        print(time.strftime("%H:%M"), "GPU", dev, "running" if p.poll() is None else f"finished ({p.returncode})",
              "::", tail(f"{WORK}/logs/gpu{dev}.log"))
print("exit codes:", [p.returncode for _, p in procs])

In [ ]:
# Summary -> results/summary.json, plus a comparison table
from ultralytics.utils.torch_utils import get_flops
import vgyolo
summary = {}
for name in NAMES:
    d = {}
    for f in sorted(glob.glob(f"{WORK}/results/{name}*/metrics.json")):
        d[f.split('/')[-2].replace(name, '') or 'all'] = json.load(open(f))
    info = glob.glob(f"{WORK}/runs/{name}/run_info.json")
    if info: d["run_info"] = json.load(open(info[0]))
    W = glob.glob(f"{WORK}/runs/{name}/weights/best.pt")
    if W:
        m = YOLO(W[0]).model
        d["params_M"] = round(sum(p.numel() for p in m.parameters()) / 1e6, 3)
        d["gflops_640"] = round(get_flops(m, 640), 1)
        vg = getattr(m.model[-1], "vg", None)
        if vg is not None and vg.fuse is not None:
            d["gamma_P2_P5"] = [round(float(f.gamma()), 4) for f in vg.fuse]
        vba = getattr(m.model[-1], "vba", None)
        if vba is not None:   # did the module switch on? gates start at 0
            d["vba_gates_P2_P5"] = [round(float(g), 4) for g in vba.gate_bias]
    summary[name] = d
os.makedirs(f"{WORK}/results", exist_ok=True)
json.dump(summary, open(f"{WORK}/results/summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

ref = {"P2 base s0": {"all": (0.8762, 0.5111), "_light": (0.9226, 0.3340), "_heavy": (0.8337, 0.5117), "_severe": (0.7543, 0.5311)},
       "vg s0":       {"all": (0.8742, 0.5161), "_light": (0.9221, 0.3419), "_heavy": (0.8254, 0.5212), "_severe": (0.7360, 0.5477)},
       "asff s0":     {"all": (0.8780, 0.5028), "_light": (0.9246, 0.3245), "_heavy": (0.8355, 0.4974), "_severe": (0.7571, 0.5229)},
       "P2 vw1 s0":      {"all": (0.8762, 0.5090), "_light": (0.9209, 0.3382), "_heavy": (0.8388, 0.4896), "_severe": (0.7768, 0.4961)},
       "P2 vw1 s1":      {"all": (0.8761, 0.5115), "_light": (0.9211, 0.3428), "_heavy": (0.8387, 0.4925), "_severe": (0.7697, 0.5042)},
       "P2 base s1": {"all": (0.8780, 0.5048), "_light": (0.9251, 0.3302), "_heavy": (0.8357, 0.4986), "_severe": (0.7515, 0.5303)}}
cols = ["P2 base s0", "P2 base s1", "P2 vw1 s0", "P2 vw1 s1"] + NAMES
print("\nAP / MR in percent (AP higher = better, MR lower = better)")
print("group   " + "".join(f"{c[:18]:>20s}" for c in cols))
for g in ["all", "_light", "_heavy", "_severe"]:
    row = f"{g.strip('_'):8s}"
    for c in cols:
        v = ref[c][g] if c in ref else ((summary[c][g]["AP"], summary[c][g]["MR"]) if g in summary[c] else None)
        row += f"{v[0]*100:12.2f}/{v[1]*100:6.2f}" if v else f"{'--':>20s}"
    print(row)